# 🎓 Student Dropout — Phase 4: Risk Classification & Scoring

**Input:** `../models/best_model.pkl`, `../data/processed/processed_dataset.csv`  

**Goal:** Use the model's continuous dropout probability to assign each enrolled student an actionable risk tier instead of a hard binary prediction.

**Tiers:**
| Tier | Probability Range |
|------|------------------|
| Low | 0–25% |
| Moderate | 25–50% |
| High | 50–75% |
| Critical | 75–100% |

## 1. Imports & Load Artifacts

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

# ── Load saved model and metadata ────────────────────────────────────────────
best_estimator = joblib.load('../models/best_model.pkl')
metadata       = joblib.load('../models/model_metadata.pkl')

print(f"Model loaded  : {metadata['model_name']}")
print(f"Features      : {len(metadata['feature_names'])}")
print(f"CV Recall     : {metadata['dropout_recall_cv']:.4f}")
print(f"CV Precision  : {metadata['dropout_precision_cv']:.4f}")

Model loaded  : XGBoost
Features      : 39
CV Recall     : 0.9471
CV Precision  : 0.5727


## 2. Reconstruct X_enrolled

In [2]:
df = pd.read_csv('../data/processed/processed_dataset.csv')

UNSCALED_TO_DROP = [
    'Age at enrollment',
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (grade)',
    'Unemployment rate',
    'Inflation rate',
    'GDP',
    'semester_performance_delta'
]
TARGET_COLS = ['Target', 'Target_Encoded', 'Target_Binary']

X = df.drop(columns=TARGET_COLS + UNSCALED_TO_DROP)

enrolled_mask = df['Target'] == 'Enrolled'
X_enrolled    = X[enrolled_mask].reset_index(drop=True)

# Keep a copy of the enrolled rows from df for context (e.g. original index)
df_enrolled = df[enrolled_mask].reset_index(drop=True)

print(f'Enrolled students : {X_enrolled.shape[0]}')

Enrolled students : 794


## 3. Phase 4 — Risk Classification & Scoring

In [3]:
# ── Step 1: Extract dropout probabilities ────────────────────────────────────
# predict_proba returns [P(Graduate), P(Dropout)]; index 1 = dropout probability
dropout_proba = best_estimator.predict_proba(X_enrolled)[:, 1]

print(f'Probability range : {dropout_proba.min():.4f}  →  {dropout_proba.max():.4f}')
print(f'Mean probability  : {dropout_proba.mean():.4f}')
print()

# ── Step 2: Map probabilities to risk tiers via pd.cut ───────────────────────
# Left edge set to -0.001 so the first bin (-0.001, 0.25] captures all values
# from 0.0 upward without relying on include_lowest edge-case behaviour.
# right=True (default) means each bin is (left, right], so:
#   Low      : 0.00 – 0.25
#   Moderate : 0.25 – 0.50
#   High     : 0.50 – 0.75
#   Critical : 0.75 – 1.00
risk_tiers = pd.cut(
    dropout_proba,
    bins=[-0.001, 0.25, 0.50, 0.75, 1.001],
    labels=['Low', 'Moderate', 'High', 'Critical'],
)

# Sanity check — no student should be unclassified
n_null = risk_tiers.isna().sum()
if n_null > 0:
    print(f'WARNING: {n_null} students could not be classified — check probability range.')

# ── Step 3: Attach to enrolled students dataframe ────────────────────────────
df_enrolled['dropout_probability'] = dropout_proba
df_enrolled['risk_tier']           = risk_tiers

# ── Step 4: Distribution summary ─────────────────────────────────────────────
tier_order = ['Critical', 'High', 'Moderate', 'Low']
distribution = (
    df_enrolled['risk_tier']
    .value_counts()
    .reindex(tier_order)
    .fillna(0)
    .astype(int)
)

print('Risk Tier Distribution')
print('=' * 45)
print(f'{"Tier":<12} {"Count":>6}  {"Pct":>6}   Bar')
print('-' * 45)
for tier, count in distribution.items():
    pct = count / len(df_enrolled) * 100
    bar = '█' * int(pct / 2)
    print(f'{tier:<12} {count:>6}  ({pct:5.1f}%)  {bar}')
print('=' * 45)
print(f'{"TOTAL":<12} {len(df_enrolled):>6}')

# ── Step 5: Preview top Critical students ────────────────────────────────────
print('\nTop 10 highest-risk enrolled students:')
cols_to_show = ['dropout_probability', 'risk_tier']
print(
    df_enrolled[cols_to_show]
    .sort_values('dropout_probability', ascending=False)
    .head(10)
    .to_string()
)

Probability range : 0.3157  →  0.9308
Mean probability  : 0.6791

Risk Tier Distribution
Tier          Count     Pct   Bar
---------------------------------------------
Critical        304  ( 38.3%)  ███████████████████
High            345  ( 43.5%)  █████████████████████
Moderate        145  ( 18.3%)  █████████
Low               0  (  0.0%)  
TOTAL           794

Top 10 highest-risk enrolled students:
     dropout_probability risk_tier
14              0.930812  Critical
352             0.930812  Critical
390             0.930812  Critical
509             0.930812  Critical
497             0.930812  Critical
167             0.930812  Critical
29              0.930812  Critical
553             0.930812  Critical
32              0.930812  Critical
706             0.930812  Critical
